# LoRA SFT v6 — MoE Weight Tying + Expanded LoRA Targets

**Goal:** Test the architectural changes from Tong Huikang's 0.85 LB methodology that we believe explain most of the 0.58 → 0.85 gap. v6 uses v3's training data (4681 records, UNCHANGED) so the ONLY variable is the LoRA/training architecture. If v6 lifts significantly, we know the gap was architectural; if flat, the gap is data or methodology.

**Architectural changes vs v3 (all from Tong's setup):**

| Param | v3 | v6 | Why |
|---|---|---|---|
| `lora_alpha` | 16 (r/2) | 32 (= r) | 2× update magnitude, aligns with reasoning-model norm |
| `lora_dropout` | 0.05 | 0.0 | Lower regularization, more signal |
| `target_modules` | in/out/up/down_proj (4) | q/k/v/o + in/out/up/down + lm_head (9) | More LoRA capacity, including attention + output projection |
| AdamW betas | (0.9, 0.999) default | (0.9, 0.95) | Modern reasoning convention (lower 2nd-moment EMA) |
| **MoE expert weight tying** | ❌ off | ✅ on | **The big one** — see below |

**Why MoE tying is the likely critical change:**

Nemotron 3 Nano has 128 MoE experts. With per-expert LoRA, each forward pass activates ~2-4 experts (top-k routing). Without tying, the other 124-126 experts' LoRA params get NO gradient signal on that example. After 585 training steps (v3), each individual expert's LoRA saw maybe ~18-22 gradient updates total — far too few to learn.

Tying means we treat the 128 expert LoRAs as ONE shared LoRA: every training example updates the SAME parameters. Effective signal density per LoRA parameter goes up ~128×. This is what Tong calls "Tinker-style" convention.

Implementation: identify LoRA params on the expert dim (shape `[128, ...]`), initialize tied (mean across dim 0), and SUM gradients across dim 0 before each optimizer step. Saved adapter naturally emits 128 per-expert copies (with identical values) — submission format unchanged.

**Data:** identical to v3 — `training-data.jsonl` (4681 records). NO new synthetic data, NO data perturbation. Strict architecture-only variable test.

**Training:** lr=2e-4, 1 epoch, effective batch 8 (same as v3). Expected steps: ~585. Expected walltime: ~6-7 hours (more LoRA params = slower per step).

**Reference points:**
- Untrained baseline: 0.500
- v3 (4 LoRA modules, no tying): **0.580** ← our floor
- v4 (gravity format fix attempt): 0.560 (regressed)
- v5 (v3 data + synth-tr): in progress
- Tong (9 modules + MoE tying + Unsloth + 8k context + ...): 0.850

**Expected lift if MoE tying alone matters:** +0.05 to +0.15. If it's actually the seq_len/Unsloth doing the work, v6 might only get +0.02-0.05. The diagnostic value either way is high.

In [ ]:
# Cell 1: Setup — sitedir discovery + ptxas-blackwell workaround + seeds
import os, random, shutil, site, stat, sys, time
import numpy as np
import torch

T0 = time.perf_counter()
def step(label):
    print(f"[{time.perf_counter() - T0:7.1f}s] {label}", flush=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Find python_packages from attached utility scripts
for utility_root in ("/kaggle/input/notebooks", "/kaggle/usr/lib/notebooks", "/kaggle/lib"):
    if not os.path.isdir(utility_root):
        continue
    for root, dirs, _ in os.walk(utility_root):
        if "models" in root.split(os.sep):
            dirs[:] = []
            continue
        if os.path.basename(root) == "python_packages":
            site.addsitedir(root)

# ptxas-blackwell exec workaround (Triton needs an executable copy)
PTXAS_DST = "/tmp/ptxas-blackwell"
for src in (
    "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell",
    "/kaggle/input/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell",
):
    if os.path.exists(src):
        if not os.path.exists(PTXAS_DST):
            shutil.copy(src, PTXAS_DST)
            os.chmod(PTXAS_DST, stat.S_IRWXU | stat.S_IRGRP | stat.S_IXGRP | stat.S_IROTH | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_PATH"] = PTXAS_DST
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = PTXAS_DST
        break

step(f"Setup done; seeds set to {SEED}")

In [ ]:
# Cell 2: Load model + tokenizer + apply LoRA (v6 config)
import kagglehub
import mamba_ssm  # noqa: F401
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
step("Tokenizer loaded")

IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
step(f"IM_END_ID={IM_END_ID}; eos_token_id={tokenizer.eos_token_id}")

t = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, device_map="auto", trust_remote_code=True, dtype=torch.bfloat16,
)
step(f"Model loaded in {time.perf_counter() - t:.0f}s, VRAM={torch.cuda.memory_allocated() / 1e9:.1f} GB")

# v6 LoRA config: alpha=r (vs v3's r/2), dropout=0 (vs v3's 0.05),
# expanded target_modules including attention proj + lm_head.
lora_cfg = LoraConfig(
    r=32,
    lora_alpha=32,                  # v3 used 16
    lora_dropout=0.0,               # v3 used 0.05
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # NEW: attention
        "in_proj", "out_proj",                    # Mamba projections (had these)
        "up_proj", "down_proj",                   # FFN/MoE projections (had these)
        "lm_head",                                # NEW: output projection
    ],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
trainable, total = sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())
step(f"LoRA wrapped: {trainable:,} trainable / {total:,} total ({100*trainable/total:.2f}%)")

# Enable gradient checkpointing to save memory
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
step("Gradient checkpointing on")

In [ ]:
# Cell 2.5: MoE expert weight tying setup
# Nemotron 3 Nano has 128 MoE experts. PEFT creates one LoRA per expert (shape
# starts with [128, ...]). With top-k routing, each forward pass activates only
# 2-4 experts, so each per-expert LoRA gets very few gradient updates.
# Tying makes all 128 expert LoRAs share the SAME values — gradient signal is
# pooled across all training examples, ~128x more effective updates per param.
# Convention (Tinker-style):
#   - For w1-side projections (up_proj/gate_proj): tie the A side
#   - For w2-side projections (down_proj):         tie the B side
#
# IMPORTANT: this assumes PEFT wrapped the MoE expert weights as a single batched
# tensor with shape [num_experts, dim_in, dim_out]. If PEFT instead created 128
# SEPARATE LoRA modules (one per expert), no tying will happen — we'll print a
# diagnostic and proceed with non-tied training (other v6 architectural changes
# still apply: alpha=32, dropout=0, expanded targets, betas).

# First, diagnostic — print a sample of LoRA params so we understand the shape layout
sample_lora_names = []
for name, param in model.named_parameters():
    if not param.requires_grad or ".lora_" not in name:
        continue
    if "experts" in name and len(sample_lora_names) < 8:
        sample_lora_names.append((name, tuple(param.shape)))
print(f"Sample expert-LoRA param shapes (for diagnostic):")
for name, shape in sample_lora_names:
    print(f"  {name}  shape={shape}")
if not sample_lora_names:
    print("  (no params with 'experts' in name)")

def find_moe_tied_params(model):
    """Return list of LoRA params that should be tied across MoE experts."""
    w1_names = ("up_proj", "gate_up_proj", "gate_proj", ".w1.")
    w2_names = ("down_proj", ".w2.")
    tied = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if ".experts." not in name or ".lora_" not in name:
            continue
        is_w1 = any(p in name for p in w1_names)
        is_w2 = any(p in name for p in w2_names)
        is_A  = ".lora_A." in name
        is_B  = ".lora_B." in name
        should_tie = (is_w1 and is_A) or (is_w2 and is_B)
        if not should_tie:
            continue
        # Need at least dim 0 = expert dim, and >1 expert
        if param.dim() < 2 or param.shape[0] <= 1:
            continue
        tied.append((name, param))
    return tied

def init_moe_tied(tied_params):
    """Start with all expert slices equal (mean across expert dim)."""
    with torch.no_grad():
        for _name, p in tied_params:
            mean = p.data.mean(dim=0, keepdim=True)
            p.data.copy_(mean.expand_as(p.data))

moe_tied = find_moe_tied_params(model)
print(f"\nMoE tying: identified {len(moe_tied)} expert-LoRA params to tie")

USING_MOE_TIE = len(moe_tied) > 0
if USING_MOE_TIE:
    for name, p in moe_tied[:5]:
        print(f"  example: {name}  shape={tuple(p.shape)}")
    if len(moe_tied) > 5:
        print(f"  ... and {len(moe_tied) - 5} more")
    init_moe_tied(moe_tied)
    step(f"MoE expert LoRAs initialized in tied state ({len(moe_tied)} param groups)")
else:
    print("⚠️ No batched expert LoRA params found. PEFT may have created per-expert")
    print("   separate modules instead of one batched tensor. v6 will proceed with")
    print("   the other architectural changes (alpha=32, dropout=0, expanded targets,")
    print("   betas) but WITHOUT MoE tying. Tying would require a custom LoRA impl.")
    step("MoE tying skipped (no batched expert params)")

from transformers import TrainerCallback

class MoETieGradsCallback(TrainerCallback):
    """Sum gradients across the expert dim before each optimizer step.

    Why SUM not MEAN: if W is the shared LoRA factor and each expert has copy
    W_i = W, then dL/dW = sum_i dL/dW_i (chain rule). Sum keeps the 128 slices
    identical after each AdamW step; mean would underweight the gradient by 128x.
    """
    def __init__(self, tied_params):
        self.tied_params = [p for _name, p in tied_params]

    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        with torch.no_grad():
            for p in self.tied_params:
                if p.grad is None:
                    continue
                grad_sum = p.grad.sum(dim=0, keepdim=True)
                p.grad.copy_(grad_sum.expand_as(p.grad))

# Build the callback list — only include MoE callback if we found tied params
trainer_callbacks = []
if USING_MOE_TIE:
    trainer_callbacks.append(MoETieGradsCallback(moe_tied))
    step("MoETieGradsCallback registered")
else:
    step("No MoE callback registered (tying inactive)")

In [ ]:
# Cell 3: Load training data + tokenize (v3 data — UNCHANGED for clean variable test)
import json
import glob

PROMPT_SUFFIX = (
    "\nPlease put your final answer inside `\\boxed{}`. "
    "For example: `\\boxed{your answer}`"
)

# v6 deliberately uses v3's training-data.jsonl (4681 records) so the only
# variable changed from v3 is the LoRA/training architecture. NOT v5's
# training-data-v5.jsonl (which adds synthetic transformation_rules).
CANDIDATE_PATHS = [
    "/kaggle/input/datasets/gideonnazarian/nemotron-solver-training-data/training-data.jsonl",
    "/kaggle/input/nemotron-solver-training-data/training-data.jsonl",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    hits = glob.glob("/kaggle/input/**/training-data.jsonl", recursive=True)
    if hits:
        DATA_PATH = hits[0]
if DATA_PATH is None:
    raise FileNotFoundError(
        f"training-data.jsonl not found. /kaggle/input/ contains: {os.listdir('/kaggle/input')}"
    )
step(f"Using DATA_PATH={DATA_PATH}")

records = []
with open(DATA_PATH) as f:
    for line in f:
        records.append(json.loads(line))
step(f"Loaded {len(records)} training records")

from collections import Counter
cats = Counter(r["category"] for r in records)
print("Records by category:")
for c, n in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"  {c:25s}: {n}")

MAX_SEQ_LEN = 1024

def tokenize_example(record):
    user_text = record["prompt"] + PROMPT_SUFFIX
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        tokenize=False, add_generation_prompt=True,
    )
    prompt_ids = tokenizer.encode(prompt_text, add_special_tokens=False)
    response_text = (
        record["reasoning"] + "\n</think>\n\\boxed{" + record["answer"] + "}<|im_end|>"
    )
    response_ids = tokenizer.encode(response_text, add_special_tokens=False)
    full_ids = prompt_ids + response_ids
    if len(full_ids) > MAX_SEQ_LEN:
        return None
    labels = [-100] * len(prompt_ids) + response_ids
    return {
        "input_ids": full_ids,
        "labels": labels,
        "attention_mask": [1] * len(full_ids),
    }

examples = []
skipped = 0
for rec in records:
    ex = tokenize_example(rec)
    if ex is None:
        skipped += 1
    else:
        examples.append(ex)
step(f"Tokenized {len(examples)} examples ({skipped} skipped > {MAX_SEQ_LEN} tokens)")

lens = [len(ex["input_ids"]) for ex in examples]
step(f"Token-length stats: min={min(lens)}, median={sorted(lens)[len(lens)//2]}, max={max(lens)}")

ex0 = examples[0]
print("\n=== Example 0 sanity check ===")
prompt_part = tokenizer.decode([t for t, l in zip(ex0["input_ids"], ex0["labels"]) if l == -100])
response_part = tokenizer.decode([l for l in ex0["labels"] if l != -100])
print(f"PROMPT (loss masked, last 200 chars): ...{prompt_part[-200:]!r}")
print(f"RESPONSE (training target, first 300 chars): {response_part[:300]!r}")

In [ ]:
# Cell 4: Train with HF Trainer + (optional) MoE tying callback
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class SFTDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, idx):
        return self.examples[idx]

def collate(batch):
    max_len = max(len(ex["input_ids"]) for ex in batch)
    input_ids, labels, attn = [], [], []
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    for ex in batch:
        n_pad = max_len - len(ex["input_ids"])
        input_ids.append(ex["input_ids"] + [pad_id] * n_pad)
        labels.append(ex["labels"] + [-100] * n_pad)
        attn.append(ex["attention_mask"] + [0] * n_pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attn, dtype=torch.long),
    }

train_ds = SFTDataset(examples)

args = TrainingArguments(
    output_dir="/kaggle/working/lora_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="no",
    bf16=True,
    optim="adamw_torch",
    adam_beta1=0.9,                # default but explicit for clarity
    adam_beta2=0.95,               # v3 used default 0.999; Tong uses 0.95
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
    gradient_checkpointing=False,  # already enabled on model
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=collate,
    callbacks=trainer_callbacks,   # MoE tie callback included only if tying active
)

step(f"Starting training (v6: alpha=32, dropout=0, 9 target modules, MoE tied={USING_MOE_TIE})")
trainer.train()
step("Training done")

In [ ]:
# Cell 5: Save adapter to /kaggle/working and zip submission
OUTPUT_DIR = "/kaggle/working"

# Clean up any stale adapter files from prior runs
for f in os.listdir(OUTPUT_DIR):
    if f.startswith("adapter") or f.endswith(".safetensors") or f.endswith(".bin"):
        try:
            os.remove(os.path.join(OUTPUT_DIR, f))
        except (IsADirectoryError, PermissionError):
            pass

# Save trained adapter
model.save_pretrained(OUTPUT_DIR)
step("Adapter saved")

# Move any nested files up to /kaggle/working level (Trainer may have nested under lora_output/)
import subprocess
subprocess.run("find /kaggle/working -name 'adapter*' -exec mv {} /kaggle/working/ \\;", shell=True, check=False)
subprocess.run("ls -la /kaggle/working/", shell=True, check=False)

# Zip everything for submission
subprocess.run("cd /kaggle/working && rm -rf lora_output && zip -m submission.zip adapter*", shell=True, check=True)
step("submission.zip created")

In [ ]:
step(f"Total wall time: {(time.perf_counter() - T0) / 60:.1f} min")
print("Done — ready for Submit to Competition.")